# 노트북 5 — 결과 분석

**EEG Resonate Encoding SNN 프로젝트** · 노트북 5

노트북 4(코랩)가 만든 `train_compare_results.csv`를 불러와 조건별 정확도를 비교하고,
정확도–스파이크 수 트레이드오프(발표 핵심 그림)를 그리며, 대응표본 Wilcoxon 검정과 대역 정렬 비교로 가설을 검증한다.

### 학습 목표
- 조건별 검증 정확도를 피험자 평균 ± 표준편차로 요약한다.
- 정확도–스파이크 수 산점도로 트레이드오프를 확인한다(H1, H2).
- 스파이크 수를 에너지 프록시(pJ)로 환산한다.
- 대응표본 Wilcoxon 검정으로 E4-정렬이 E3-정렬보다 유의미하게 나은지 확인한다.
- E4의 대역 배치 3조건(정렬/균등/어긋남)을 비교해 대역 정렬 효과를 확인한다(H3).


## 1. 설치와 임포트

In [ ]:
%pip install -q scipy

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import koreanize_matplotlib          # 그래프 한글 폰트

np.random.seed(0)


## 2. 결과 CSV 불러오기

노트북 4가 저장한 조건×피험자×시드 단위 결과를 불러온다. 시드 2개는 같은 (조건,피험자)의 반복측정으로 보고
평균내어 피험자 단위 값 하나로 줄인다(대응표본 검정은 피험자 단위로 짝짓는다).

In [ ]:
CONDITION_ORDER = ['E1', 'E2', 'E3_aligned', 'E4_aligned', 'E4_uniform', 'E4_misaligned']
IN_PATH = '../data/processed/train_compare_results.csv'

raw_df = pd.read_csv(IN_PATH)
raw_df['condition'] = pd.Categorical(raw_df['condition'], categories=CONDITION_ORDER, ordered=True)

# 시드 2개를 (조건,피험자) 단위로 평균내 피험자 단위 값 하나로 축약
subj_df = raw_df.groupby(['condition', 'subject_id'], as_index=False, observed=True)[
    ['val_acc', 'spikes_per_trial']
].mean()

print('전체 실행 수:', len(raw_df), '| 피험자 단위 행 수:', len(subj_df))
subj_df.head()


## 3. 조건별 검증 정확도 요약

피험자 평균 ± 표준편차로 조건을 비교한다(H1: E4 ≥ E3 > E2 ≈ E1 예상).

In [ ]:
summary = subj_df.groupby('condition', observed=True)['val_acc'].agg(['mean', 'std']).reset_index()
summary = summary.set_index('condition').loc[CONDITION_ORDER].reset_index()   # 순서 고정
summary['mean_pct'] = summary['mean'] * 100
summary['std_pct'] = summary['std'] * 100

print(summary[['condition', 'mean_pct', 'std_pct']])

plt.figure(figsize=(8, 4))
plt.bar(summary['condition'], summary['mean_pct'], yerr=summary['std_pct'], capsize=4, color='tab:blue')
plt.axhline(50, color='gray', linestyle='--', linewidth=1, label='우연 수준(50%)')
plt.ylabel('검증 정확도 (%)')
plt.title('조건별 검증 정확도 (피험자 평균 ± 표준편차)')
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout(); plt.show()


## 4. 정확도–스파이크 수 트레이드오프 (발표 핵심 그림)

같은 정확도에서 E4가 E3보다 스파이크 수가 적으면(왼쪽에 위치) H2가 지지된다.

In [ ]:
condition_colors = {
    'E1': 'tab:blue', 'E2': 'tab:orange', 'E3_aligned': 'tab:green',
    'E4_aligned': 'tab:red', 'E4_uniform': 'tab:purple', 'E4_misaligned': 'tab:brown',
}

plt.figure(figsize=(7, 5))
for condition_name, color in condition_colors.items():
    sub = subj_df[subj_df['condition'] == condition_name]
    plt.scatter(sub['spikes_per_trial'], sub['val_acc'] * 100, label=condition_name, color=color, alpha=0.8)

plt.axhline(50, color='gray', linestyle='--', linewidth=1)
plt.xlabel('추론 1회당 총 스파이크 수 (입력+은닉+출력)')
plt.ylabel('검증 정확도 (%)')
plt.title('정확도–스파이크 수 트레이드오프')
plt.legend()
plt.tight_layout(); plt.show()


## 5. 에너지 프록시 환산

SNN 누적연산(AC) 1회당 0.9 pJ로 가정해 시행당 총 에너지를 추정한다(45 nm CMOS 기준).
참고용으로 ANN MAC 연산은 1회당 4.6 pJ로, SNN이 훨씬 적은 에너지만으로 동작함을 높이는 데 쓴다(직접 비교용 ANN 모델은 구현하지 않았다).

In [ ]:
AC_ENERGY_PJ = 0.9     # SNN 누적연산 1회당 에너지 [pJ]
MAC_ENERGY_PJ = 4.6    # ANN MAC 1회당 에너지 [pJ] (참고용, 45nm CMOS)

subj_df['energy_pJ'] = subj_df['spikes_per_trial'] * AC_ENERGY_PJ

energy_summary = subj_df.groupby('condition', observed=True)['energy_pJ'].mean()
energy_summary = energy_summary.loc[CONDITION_ORDER]

for condition_name, energy in energy_summary.items():
    print(f'{condition_name}: 시행당 평균 에너지 {energy:.1f} pJ (스파이크 {energy/AC_ENERGY_PJ:.0f}개)')


## 6. 대응표본 Wilcoxon 검정 (E4-정렬 vs E3-정렬)

같은 피험자·같은 분할이므로 대응표본 검정을 쓴다. E3, E4의 선형 부분이 동일하므로(CLAUDE.md 2절),
여기서 차이가 있다면 스파이크 생성 규칙(비선형 부분)의 효과로 해석한다.

In [ ]:
e4_aligned = subj_df[subj_df['condition'] == 'E4_aligned'].sort_values('subject_id')['val_acc'].to_numpy()
e3_aligned = subj_df[subj_df['condition'] == 'E3_aligned'].sort_values('subject_id')['val_acc'].to_numpy()

stat, p_value = wilcoxon(e4_aligned, e3_aligned)
print(f'Wilcoxon 검정 (E4-정렬 vs E3-정렬): 통계량={stat:.3f}, p={p_value:.4f}')
print(f'E4-정렬 평균 정확도: {e4_aligned.mean()*100:.1f}% | E3-정렬 평균 정확도: {e3_aligned.mean()*100:.1f}%')

verdict = np.array(['유의미한 차이 없음 (p≥0.05)', '유의미한 차이 있음 (p<0.05)'])[int(p_value < 0.05)]
print('결과:', verdict)


## 7. 대역 정렬 3조건 비교 (E4: 정렬/균등/어긋남)

H3(대역 정렬 효과)는 원래 E3와 E4 양쪽에서 정렬/균등/어긋남을 비교해야 하지만,
실행 조건(CLAUDE.md 3절)이 E3는 정렬 하나만 포함하므로 이 노트북에서는 **E4의 3조건만** 비교한다.
E3의 대역 배치 절제까지 비교하려면 노트북 4에 E3_uniform, E3_misaligned 조건을 추가로 돌려야 한다(한계로 명시).

In [ ]:
band_conditions = ['E4_aligned', 'E4_uniform', 'E4_misaligned']
band_summary = summary[summary['condition'].isin(band_conditions)]

plt.figure(figsize=(6, 4))
plt.bar(band_summary['condition'], band_summary['mean_pct'], yerr=band_summary['std_pct'],
        capsize=4, color=['tab:red', 'tab:purple', 'tab:brown'])
plt.axhline(50, color='gray', linestyle='--', linewidth=1)
plt.ylabel('검증 정확도 (%)')
plt.title('E4 대역 배치 절제 — 정렬 vs 균등 vs 어긋남')
plt.tight_layout(); plt.show()

aligned_acc = subj_df[subj_df['condition'] == 'E4_aligned']['val_acc'].mean()
misaligned_acc = subj_df[subj_df['condition'] == 'E4_misaligned']['val_acc'].mean()
alignment_effect = (aligned_acc - misaligned_acc) * 100
print(f'E4 대역 정렬 효과 (정렬 − 어긋남): {alignment_effect:+.1f}%p (양수면 정렬 조건이 어긋남보다 우수 — 가설과 일치)')


## 정리

- `train_compare_results.csv`를 불러와 시드 2개를 평균내어 피험자 단위 값으로 줄인 뒤, 조건별 검증 정확도(평균±표준편차)를 비교했다.
- 정확도–스파이크 수 산점도(발표 핵심 그림)로 H1, H2를 확인했고, 스파이크 수를 pJ 단위 에너지로 환산했다.
- 대응표본 Wilcoxon 검정으로 E4-정렬과 E3-정렬의 차이가 통계적으로 유의미한지 확인했다.
- E4의 대역 배치 3조건(정렬/균등/어긋남)을 비교해 대역 정렬 효과를 확인했다(H3은 E3 측의 대역 조건이 없어 부분적으로만 검증됨).

이제 이 그림들과 검정 결과를 바탕으로 발표자료(11절 구성)를 만든다.